# Stimulus check: how much does SHINE change the pixel and CLIP RDMs?

Does SHINE change the *similarity structure of the stimuli themselves*? No participants and no reference RDM (in particular no Kiani hierarchy) are involved: for each feature space, the pre-SHINE RDM is compared only with its own post-SHINE RDM.<br>
Two feature spaces: **pixels** (flattened RGB, Euclidean, `sens_pre` / `sens_post`) and **CLIP ViT-B/32** (embedding cosine, `clip_pre` / `clip_post`).<br>
Companion notebooks compare the *cohorts' behavior* (`1_` to `6_`); this one is the manipulation check underneath them. Backend: `sensory.py`; RDMs come from `python -m analysis.s0_build_rdms.build_all`.

**Inference.** Both RDMs are deterministic functions of the 725 images, so there is no participant noise to bootstrap. The uncertainty is which images were sampled, so CIs come from an **image bootstrap**: images are resampled with replacement and every statistic is recomputed over the resampled pairs (a pair of an image with its own duplicate is dropped). This respects the dependence between the 262,450 pairs, which a pair-level bootstrap would ignore. There is no hypothesis test: "are the RDMs identical?" is not a question worth a p-value, the size of the difference is.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve().parents[1]  # analysis/s1_pre_vs_post_shine/ -> repo root
sys.path.insert(0, str(REPO_ROOT))

from analysis.s0_build_rdms.common import load_rdm
from analysis.s1_pre_vs_post_shine import sensory
from analysis.s1_pre_vs_post_shine.sensory import SENSORY_CACHE_DIR, SPACE_LABEL

import plotly.io as pio
pio.renderers.default = "notebook_connected"  # figures load plotly.js once instead of embedding it

In [2]:
SPACES = ["sens", "clip"]
RDMS = {s: (load_rdm(f"{s}_pre"), load_rdm(f"{s}_post")) for s in SPACES}
SEED, N_BOOT = 0, 1000
print({s: f"{len(RDMS[s][0]):,} pairs" for s in SPACES})

{'sens': '262,450 pairs', 'clip': '262,450 pairs'}


## 1. How similar are the two RDMs?

- **Kendall tau-a** and **Spearman rho**: do the two RDMs order the image pairs the same way? (tau-a is the primary rank statistic throughout the project; ties are not removed from its denominator.)
- **Pearson r** on the raw distances: do they agree *linearly*, not just in rank?
- **Mean distance, post / pre**: did SHINE change the overall scale of the space? Rank statistics cannot see this.

In [3]:
boot = {s: sensory.bootstrap_rdm_agreement(*RDMS[s], n_boot=N_BOOT, seed=SEED,
                                           cache_path=SENSORY_CACHE_DIR / f"{s}_bootstrap.npz") for s in SPACES}
tables = {s: sensory.agreement_table(*RDMS[s], boot[s]) for s in SPACES}
for s in SPACES:
    print(f"{SPACE_LABEL[s]}  ({N_BOOT} image-bootstrap draws)")
    print(tables[s].round(3).to_string(), "\n")
sensory.fig_agreement_forest(tables).show()

pixels (Euclidean)  (1000 image-bootstrap draws)
                           estimate  ci_low  ci_high
statistic                                           
Kendall tau-a                 0.534   0.508    0.562
Spearman rho                  0.728   0.699    0.758
Pearson r                     0.728   0.700    0.756
mean distance, post / pre     1.002   0.990    1.013 

CLIP ViT-B/32 (cosine)  (1000 image-bootstrap draws)
                           estimate  ci_low  ci_high
statistic                                           
Kendall tau-a                 0.764   0.751    0.777
Spearman rho                  0.921   0.912    0.929
Pearson r                     0.933   0.925    0.940
mean distance, post / pre     0.995   0.989    1.001 



**Scale of the numbers.** For reference only: the two *behavioral* cohorts (`3_geometry.ipynb`) agree with each other at tau 0.15 and rho 0.22 on their raw RDMs. That is a different quantity (two noisy samples of human judgments, partly covering the pairs), so it is not a benchmark for the values above; it is cited only so that "0.7" and "0.9" can be read against something.

## 2. Where do they disagree?

Every image pair as one point: its distance before SHINE against its distance after. Points on the dashed identity line are untouched; spread around it is SHINE reorganizing the space. A bend away from the line would be a nonlinear distortion that Pearson and Spearman summarise differently.

In [4]:
sensory.fig_pre_post_density(RDMS).show()

## 3. Which images moved?

The RDM-level numbers can hide whether a few images changed a lot or every image changed a little. Two per-image quantities:

- **Feature shift**: the cosine (and Euclidean) distance between an image's own pre and post feature vector. For CLIP these are the stored embeddings, for pixels the flattened RGB images.
- **Row agreement**: Spearman rho between the image's row of the pre RDM and its row of the post RDM, i.e. whether its *neighbourhood* (its ranking of all 724 other images) survived.

In [5]:
shifts = {"sens": sensory.pixel_shift(SENSORY_CACHE_DIR / "pixel_shift.pkl"), "clip": sensory.clip_shift()}
per_image = {s: shifts[s].assign(row_rho=sensory.row_agreement(*RDMS[s])) for s in SPACES}

def describe(x: pd.Series) -> str:
    q1, med, q3 = x.quantile([0.25, 0.5, 0.75])
    return f"median {med:.4f}  IQR [{q1:.4f}, {q3:.4f}]  range [{x.min():.4f}, {x.max():.4f}]"

for s in SPACES:
    print(SPACE_LABEL[s])
    print("  cosine shift   ", describe(per_image[s]["cosine"]))
    print("  euclid. shift  ", describe(per_image[s]["euclidean"]))
    print("  row rho        ", describe(per_image[s]["row_rho"]))
sensory.fig_shift_vs_row_agreement(per_image).show()

pixels (Euclidean)
  cosine shift    median 0.0045  IQR [0.0017, 0.0112]  range [0.0001, 0.0574]
  euclid. shift   median 6491.9774  IQR [4077.6095, 9844.2940]  range [943.5794, 26717.2503]
  row rho         median 0.7725  IQR [0.7139, 0.8355]  range [-0.0705, 0.9627]
CLIP ViT-B/32 (cosine)
  cosine shift    median 0.0243  IQR [0.0131, 0.0464]  range [0.0017, 0.2548]
  euclid. shift   median 2.4289  IQR [1.7689, 3.3735]  range [0.6293, 7.9742]
  row rho         median 0.9248  IQR [0.8945, 0.9397]  range [0.5396, 0.9681]


In [6]:
from scipy.stats import spearmanr

for s in SPACES:
    rho = spearmanr(per_image[s]["cosine"], per_image[s]["row_rho"]).statistic
    print(f"{SPACE_LABEL[s]:26s} rho(feature shift, row rho) = {rho:+.3f}  (negative: bigger shift, worse-preserved neighbourhood)")
cross = spearmanr(per_image["sens"]["cosine"], per_image["clip"]["cosine"]).statistic
print(f"\nrho(pixel shift, CLIP shift) across images = {cross:+.3f}")

pixels (Euclidean)         rho(feature shift, row rho) = -0.342  (negative: bigger shift, worse-preserved neighbourhood)
CLIP ViT-B/32 (cosine)     rho(feature shift, row rho) = -0.423  (negative: bigger shift, worse-preserved neighbourhood)

rho(pixel shift, CLIP shift) across images = +0.666


In [7]:
cols = ["curated_path", "cosine", "row_rho"]
for s in SPACES:
    print(f"{SPACE_LABEL[s]}: 8 largest shifts")
    print(per_image[s].nlargest(8, "cosine")[cols].round(4).to_string(index=False), "\n")
print("CLIP: 8 images whose neighbourhood changed most (lowest row rho)")
print(per_image["clip"].nsmallest(8, "row_rho")[cols].round(4).to_string(index=False))

pixels (Euclidean): 8 largest shifts
                            curated_path  cosine  row_rho
   inanimate\handmade\ball\football2.png  0.0574  -0.0553
  inanimate\handmade\clothing\dress1.png  0.0505   0.5028
      inanimate\handmade\tool\wheel1.png  0.0490   0.6945
  inanimate\handmade\ball\golf_ball3.png  0.0456  -0.0077
 inanimate\handmade\ball\basketball4.png  0.0453   0.8756
inanimate\handmade\electric\speaker1.png  0.0430   0.6880
     inanimate\handmade\kitchen\mug2.png  0.0424   0.2198
       inanimate\natural\food\apple3.png  0.0420   0.2815 

CLIP ViT-B/32 (cosine): 8 largest shifts
                                  curated_path  cosine  row_rho
inanimate\natural\flower\white\calla_lily1.png  0.2548   0.5396
            inanimate\natural\food\tomato2.png  0.2015   0.8725
            inanimate\handmade\food\bread1.png  0.1697   0.7717
  inanimate\natural\flower\white\cyclamen1.png  0.1662   0.7861
           inanimate\handmade\kitchen\pot3.png  0.1550   0.5497
   inanimate\h

## 4. Takeaways

1. **SHINE reorganizes the pixel RDM substantially and the CLIP RDM mildly.** Pre vs post agreement is tau-a 0.53 [0.51, 0.56] and rho 0.73 [0.70, 0.76] for pixels, against tau-a 0.76 [0.75, 0.78] and rho 0.92 [0.91, 0.93] for CLIP. The intervals do not overlap, so CLIP's neighbourhood structure is clearly more robust to SHINE than the pixel space's.
2. **The overall scale does not change.** Mean distance post / pre is 1.002 [0.990, 1.013] for pixels and 0.995 [0.989, 1.001] for CLIP. Pearson r equals Spearman rho for pixels (0.73) and is slightly higher for CLIP (0.93 vs 0.92), so there is no sign of a nonlinear distortion beyond the rank reshuffling.
3. **Every image moves a little; a few move a lot, and moving more means a worse-preserved neighbourhood.** Median per-image CLIP cosine shift is 0.024 (max 0.25); per-image row rho is 0.92 for CLIP (lowest 0.54) and 0.77 for pixels (lowest -0.07). Shift and row rho are negatively related (rho -0.34 pixels, -0.42 CLIP), and the images that shift most in pixel space also shift most in CLIP space (rho +0.67).
4. **A handful of images drive the tail.** The largest CLIP shifts are calla_lily1 (0.25), tomato2, bread1 and cyclamen1; the pixel-space rows that break down entirely (row rho about 0) are football2 and golf_ball3. The cause is not examined here.

Scope: this is a stimulus-level description. It says how different the two *feature* spaces are, not how much the cohorts' behavior differs (see `1_` to `5_`), and the CIs are over the image sample, not over participants.